# Confessional Pipeline - The Real Homies of San Francisco
Kaggle notebook (Jupytext `# %%` -> import as .ipynb). Generates confessional video
for all 6 characters: audio (Fish Audio Pro Free) -> lip-sync (LivePortrait) ->
motion B-roll (LTX Video I2V) -> L-cut assembly. All dialogue + prompts sourced from
`confessional-kit.md` (project root). Voice IDs are placeholders; assign real Fish Audio
voice_id OR 5-sec reference WAV per character to lock timbre.

## Cell 1 - Install & Import

In [1]:
# Uncomment on first Kaggle run:
# !pip install torch torchaudio diffusers[cpu] transformers accelerate pillow
# !pip install "git+https://github.com/facebookresearch/Copy-and-Paste-Programming.git"
# !pip install openvpype-fish  # Fish Audio SDK wrapper
# !pip install "git+https://github.com/KwaiVGI/LivePortrait.git"
# !pip install opencv-python-headless
import os, json, pathlib, requests
from dataclasses import dataclass, field
from typing import List, Optional

import torch
from PIL import Image
from diffusers import LTXImageToVideoPipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| cuda:", torch.cuda.is_available())

device: cuda | cuda: True


## Cell 2 - Character roster (dialoque + voice + portrait prompts)

In [2]:
CHAR_DIR = pathlib.Path("visuals/characters")
OUT_DIR  = pathlib.Path("outputs/confessionals")
OUT_DIR.mkdir(parents=True, exist_ok=True)

@dataclass
class Character:
    key: str
    label: str
    voice_id: str            # Fish Audio voice_id or path to a reference WAV (FIXED per char)
    dialogue: List[str]     # spoken confessional beats, in order
    image_prompt: str       # base still generation prompt (standardized look)
    portrait_path: str     # local portrait after rendering Cell 9
    voice_notes: str

def _base():
    return ("Realistic photo of a gay man in a Bravo-style confessional booth: seated in a curved "
            "white neural confessional pod with soft gold LED rim strips, wearing a charcoal fitted "
            "V-neck tee and dark slim-fit jeans, barefoot. 8500K gold rim lighting from behind, "
            "front-left key fill, gold reflector, shallow bokeh background. Smart-glass wall behind "
            "is opaque black. Three-quarter profile to camera, slight chin lift, one eyebrow raised. "
            "ISO 12233 sharpness in catchlight, 8K cinematic shallow DOF. Square 800x800 aspect.")

CHARACTERS = {
    "PREST": Character(
        key="prest", label="Prest (Host/EP, 42)",
        voice_id="0f2fdfad679c454bb82be023104ba3ba",
        voice_notes="Male 42, rapid triple-clause cadence, drops consonants, cynical smirk, never says 'I don't know'.",
        dialogue=[
            "We cast seven men in this house, and only one of them understood the first rule: every villain edit is a love story someone forgot to finish. Felix thinks he broke the Bethenny Clause - he broke one contract. I've got seventeen. [beat] Now? I'm worth more than the network thinks I am.",
            "The algorithm says conflict probability between Marco and Dex is 87%. It's wrong - it's always wanting something that makes them dangerous. Dex wants security; Marco wants the bonus. The edit wants blood.",
            "People ask me if I feel guilty. Guilty? I'm the one holding the knife. The question is - which of you is brave enough to grab it back?",
        ],
        image_prompt=_base() + " 42-year-old white man, silver-flecked dark wavy hair, salt-and-pepper trimmed beard, piercing dark eyes, gold pinky ring. --ar 1:1 --style raw",
        portrait_path="https://raw.githubusercontent.com/alston808/the-real-houseboys-of-san-francisco/refs/heads/main/images/prest.png?token=GHSAT0AAAAAAEFTG25X5AB453LLSL4STCZY2WHF22A",
    ),
    "FELIX": Character(
        key="felix", label="Felix (OG Entrepreneur, 35)",
        voice_id="77974fed34614080a505a797bb96357b",
        voice_notes="Male 35, East Asian, measured lawyerly pauses, thumb press, never says 'please'.",
        dialogue=[
            "First season, they offered me seven grand to be the crazy bitch. [beat] I crossed out the profit-sharing clause before the lawyers even blinked. Now? I'm worth more than the network thinks I am.",
            "Ash asked me yesterday if the contract is legal. Honey, legality died when the NLRB ruled a dating-show contestant is an employee. We're not people with rights - we're content assets.",
            "They wanted me to drink so I'd be boring on camera. Instead I turned my sobriety into a brand. That's the plot twist they can't edit.",
        ],
        image_prompt=_base() + " 35-year-old East Asian man, sharp angular features, blunt black bob, intense eyes behind thin frames, no jewelry except steel bracelet, leaning forward calculating gaze. --ar 1:1 --style raw",
        portrait_path="https://raw.githubusercontent.com/alston808/the-real-houseboys-of-san-francisco/refs/heads/main/images/felix.png?token=GHSAT0AAAAAAEFTG25X5AB453LLSL4STCZY2WHF22Ag",
    ),
    "MARCO": Character(
        key="marco", label="Marco (Tech Heir, 34)",
        voice_id="5c8dc6a69c0b4edfb32634db6384bf34",
        voice_notes="Male 34, Mediterranean, smooth influencer three-second beats, 'literally' quantifier, smile not reaching eyes.",
        dialogue=[
            "The coupling bonus is a lie. A beautiful, expensive lie - twenty-five Gs if Dex and I stay 'romantic' for sixty days of footage. [quietly] My crypto's down thirty percent. The bonus pays the tax bill.",
            "DEX - [sighs, looks at ring finger] - the producers asked me what makes a good villain edit. I told them 'a man who loves his boyfriend but can't afford to admit it on camera.'",
            "They want drama? Give them drama. But this - this real shit with Dex? That's mine. And the edit can choke on it.",
        ],
        image_prompt=_base() + " 34-year-old Mediterranean man, dark wavy hair with highlights, light stubble, gold signet ring, gold hoop earrings, smooth practiced smile not reaching eyes. --ar 1:1 --style raw",
        portrait_path="https://raw.githubusercontent.com/alston808/the-real-houseboys-of-san-francisco/refs/heads/main/images/marco.png?token=GHSAT0AAAAAAEFTG25X5AB453LLSL4STCZY2WHF22A",
    ),
    "DEX": Character(
        key="dex", label="Dex (Young Lover, 26)",
        voice_id="638d1477f70a474eb13e0ec8ce6665c9",
        voice_notes="Male 26, white, Midwest-nice that fractures under stress, apologizes mid-sentence, rubs collarbone.",
        dialogue=[
            "My mugshot ran the season premiere teaser - [small laugh] 'Local Boy Becomes Content.' The irony? I'm the most real person here. Which means I'm already the villain.",
            "People always ask me why I joined. I said 'love.' Really? Rent was due and the stipend's a thousand a week. The NLRB said we're employees, not participants. [pause] Employees deserve minimum wage.",
            "They're gonna edit me however they want. But the raw feed? The raw feed shows me choosing Marco every morning before cameras. That's the cut they can't use.",
        ],
        image_prompt=_base() + " 26-year-old white man, porcelain skin, freckles across nose, dark curly hair, gold hoop earrings, youthful guarded expression, slight fidget. --ar 1:1 --style raw",
        portrait_path="https://raw.githubusercontent.com/alston808/the-real-houseboys-of-san-francisco/refs/heads/main/images/dex.png?token=GHSAT0AAAAAAEFTG25X5AB453LLSL4STCZY2WHF22A",
    ),
    "RIVER": Character(
        key="river", label="River (The Wildcard, 31)",
        voice_id="7a18a1851d2649108c48ec9f2c80eb2c",
        voice_notes="Male 31, Latino, switches sweet-boy to flat-angry register, twists wedding band, defiant half-smile.",
        dialogue=[
            "The casting director asked if I'd be comfortable being 'vulnerable.' I said 'sure, as long as vulnerable pays more than the last meltdown.' They booked me a first-class ticket from SLC. [wry] Soft-swinging in Utah doesn't get you redemption arcs - just body cams.",
            "A producer told me - off-the-record, which means on-the-record - that they cast two or three people as the 'don't-give-a-f***' types. 'That is decided before they even interview anyone.' Me? I'm their favorite mistake. [beat] But here's the thing about mistakes: we get to choose when the edit ends.",
            "You want a villain? You want a redemption arc? A meltdown? [looks dead into camera] I'll give you all three. Watch me rewrite the cut they can't control.",
        ],
        image_prompt=_base() + " 31-year-old Latino man, dark wavy hair, expressive dark eyes, light stubble, visible forearm tattoos under rolled sleeves of charcoal V-neck, silver hoop earrings, arms crossed, defiant half-smile. --ar 1:1 --style raw",
        portrait_path="https://raw.githubusercontent.com/alston808/the-real-houseboys-of-san-francisco/refs/heads/main/images/river.png?token=GHSAT0AAAAAAEFTG25X5AB453LLSL4STCZY2WHF22A",
    ),
    "ASH": Character(
        key="ash", label="Ash (The Outsider, 28)",
        voice_id="7f6d9042c86c4ad19595729471cb1a84",
        voice_notes="Male 28, South Asian, precise engineer phrasing, Bengali lilt, counts on fingers, 'numbers don't lie' tic.",
        dialogue=[
            "The contract says I'm a 'participant.' The NLRB called that misclassification in the Love Is Blind case - seventeen cast members paid a thousand a week while being filmed 24/7. [small precise] That's not participation. That's wage theft with better lighting.",
            "I asked about mental health support during onboarding. They said 'independent physician evaluation' and scheduled it for after filming started. [pause] My anxiety spiked 300% in the first 48 hours. The psych eval never happened.",
            "You all think you're performing for the audience. But the edit's just another algorithm - and I spent two years building recommendation engines. I know how this ends. [beat] The most authentic moment gets cut every time.",
        ],
        image_prompt=_base() + " 28-year-old South Asian man, brown skin, wire-rimmed glasses, close-cropped hair, warm brown eyes, simple silver chain, holding a small fidget toy made of braided cables, nervous expression looking at floor then forcing eye contact. --ar 1:1 --style raw",
        portrait_path="https://raw.githubusercontent.com/alston808/the-real-houseboys-of-san-francisco/refs/heads/main/images/ash.png?token=GHSAT0AAAAAAEFTG25X5AB453LLSL4STCZY2WHF22A",
    ),
}

print("Characters loaded:", list(CHARACTERS.keys()))

Characters loaded: ['PREST', 'FELIX', 'MARCO', 'DEX', 'RIVER', 'ASH']


## Cell 3 - LTX Video Image-to-Video (motion B-roll / reaction clips)

In [3]:
# LTX-Video I2V pipeline (diffusers), CPU-offloaded for Kaggle T4
pipe = LTXImageToVideoPipeline.from_pretrained(
    "Lightricks/LTX-Video",
    torch_dtype=torch.bfloat16,
)
pipe.enable_model_cpu_offload()  # sequential CPU offload -> fits 16 GB T4

# Standardized motion for a 3-s confessional beat (matches confessional-kit.md)
def ltx_motion_from_anchor(anchor_path: str, motion_prompt: str,
                           out_name: str, num_frames: int = 81, steps: int = 35,
                           guidance: float = 3.5):
    """Generate a 2-4s motion clip from a static confessional portrait.
    Used for the 2-second 'on-camera' reaction hook or cutaway B-roll."""
    init_image = Image.open(anchor_path).convert("RGB").resize((704, 480))
    video = pipe(
        image=init_image,
        prompt=motion_prompt,
        negative_prompt="morphing, twitching, warping face, extra limbs, rapid zoom, blur",
        num_frames=num_frames,
        num_inference_steps=steps,
        guidance_scale=guidance,
    ).frames[0]
    export_to_video(video, str(OUT_DIR / f"{out_name}.mp4"), fps=24)
    return str(OUT_DIR / f"{out_name}.mp4")

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


model_index.json:   0%|          | 0.00/412 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

## Cell 4 - Fish Audio voice lock (per-character determinism)
Replace the placeholder IDs with your Fish Audio voice_id or a reference WAV path.
Fixed voice_id -> 100% deterministic timbre every run.

In [4]:
FISH_VOICE_IDS = {
    "prest": "0f2fdfad679c454bb82be023104ba3ba",  # Prest — Host/EP
    "felix": "77974fed34614080a505a797bb96357b",  # Felix — OG Entrepreneur
    "marco": "5c8dc6a69c0b4edfb32634db6384bf34",  # Marco — Tech Heir
    "dex":   "638d1477f70a474eb13e0ec8ce6665c9",  # Dex — Young Lover
    "river": "7a18a1851d2649108c48ec9f2c80eb2c",  # River — Wildcard
    "ash":   "7f6d9042c86c4ad19595729471cb1a84",  # Ash — Outsider
}

FISH_AUDIO_API_KEY = "sk-fish-8WH5hkl_4SbyUgWDYEems8lkPmq_9CfKyQ8w92xDGk8"

def tts_for_character(char_key: str, text: str, out_path: str) -> str:
    """Generate speech WAV via Fish Audio Pro Free (direct API). No fallback."""
    voice_id = FISH_VOICE_IDS[char_key]
    payload  = {"text": text, "reference_id": voice_id, "format": "wav"}
    r = requests.post("https://api.fish.audio/v1/tts",
                      json=payload,
                      headers={"Authorization": f"Bearer {FISH_AUDIO_API_KEY}",
                               "Content-Type": "application/json",
                               "model": "s2.1-pro-free"},
                      timeout=300)
    r.raise_for_status()
    pathlib.Path(out_path).write_bytes(r.content)
    print(f"[fish] {char_key}: {voice_id[:8]}…  -> {out_path}")
    return out_path

## Cell 5 - LivePortrait lip-sync (audio-driven talking head)
Drives the static confessional portrait from the Fish Audio WAV.
Install on Kaggle: pip install "git+https://github.com/KwaiVGI/LivePortrait.git"

In [5]:
LIVEPORTRAIT_DIR = os.environ.get("LIVEPORTRAIT_DIR",
    "/content/LivePortrait")  # clone or pip install target

def lip_sync_portrait(char_key: str, wav_path: str, beat_index: int) -> str:
    """Run LivePortrait audio->video on the character's confessional portrait.
    Input: portrait PNG + speech WAV. Output: synced ~3s MP4 (the 'on-camera' clip)."""
    char = CHARACTERS[char_key]
    src = char.portrait_path
    out_mp4 = OUT_DIR / f"{char.key}_conf_{beat_index}.mp4"
    cmd = [
        "python", "inference.py",
        "--source_image", src,
        "--driving_audio", wav_path,
        "--output_dir", str(OUT_DIR),
        "--flag_lip_zero", "True",
    ]
    import subprocess
    full = f"{src}|{wav_path}"
    log_name = OUT_DIR / f"{char.key}_conf_{beat_index}.txt"
    with open(log_name, "w") as fh:
        subprocess.run(cmd, cwd=LIVEPORTRAIT_DIR, stdout=fh, stderr=subprocess.STDOUT,
                       check=False)
    # LivePortrait writes <src_name>_lip_sync.mp4 into output_dir
    print(f"[LivePortrait] {char.label}: {full} -> {out_mp4}")
    return str(out_mp4)

## Cell 6 - Orchestrator: full confessional for one character
Produces: 1 audio WAV + 1 lip-synced on-camera clip + 1 LTX reaction B-roll per beat.
Emits a CSV frame log consumable by any NLE (shot list + cut points for the L-cut).

In [6]:
import csv, subprocess, json

def run_confessional(char_key: str, beats: list = None, do_motion: bool = True):
    """
    Generate the full confessional packet for ONE character.
    Returns a list of clip metadata dicts + writes a frame-log CSV.
    """
    char = CHARACTERS[char_key]
    beats = beats or list(enumerate(char.dialogue))
    clips = []
    frame_log = []

    for i, line in enumerate(char.dialogue, start=1):
        wav = OUT_DIR / f"{char.key}_audio_{i}.wav"
        tts_for_character(char_key, line, str(wav))           # -> audio (Cell 4)
        on_cam = lip_sync_portrait(char_key, str(wav), i)      # -> synced on-camera MP4 (Cell 5)

        b_roll = None
        if do_motion:
            # LTX reaction clip: standardized micro-motion (head tilt + micro-expression)
            reaction_prompt = (
                "Direct confessional interview framing, static medium close-up, dramatic reality "
                "TV lighting. The man rolls his eyes subtly, sighs dramatically, takes a slow sip "
                "from a wine glass. Subtle breathing, continuous eye gaze toward camera, high-end "
                "production quality --no blur, no morphing, no extra limbs."
            )
            motion_name = f"{char.key}_reaction_{i}"
            try:
                b_roll = ltx_motion_from_anchor(char.portrait_path, reaction_prompt, motion_name)
            except Exception as e:
                print(f"[LTX motion skipped for {char.key} beat {i}: {e}]")
                b_roll = None

        clips.append({"beat": i, "line": line, "audio": str(wav),
                      "on_camera": on_cam, "b_roll": b_roll})

        # L-cut: on-camera audio continues through the CUTAWAY to b_roll
        frame_log.append({
            "char": char_key, "beat": i, "on_camera_mp4": on_cam,
            "b_roll_mp4": b_roll or "(live_portrait_only)",
            "l_cut_seconds": 2.0,   # on-camera plays 2s before audio cuts to VO over b-roll
            "audio_wav": str(wav),
            "speakers_line": textwrap.shorten(line, 70),
        })

    log_csv = OUT_DIR / f"{char.key}_framelog.csv"
    with open(log_csv, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=frame_log[0].keys())
        w.writeheader()
        w.writerows(frame_log)
    print(json.dumps(clips, indent=2)[:400] + "...")
    return clips, str(log_csv)

In [7]:
# Pre-flight checklist before running:
#  [ ] visuals/characters/prest_confessional.png exists (generated from base prompt + Prest variant)
#  [ ] FISH_API_KEY env var set OR Kokoro installed for fallback voices
#  [ ] LivePortrait installed + LIVEPORTRAIT_DIR env set if not at /content/LivePortrait
#  [ ] GPU accelerator ON (T4 x2 recommended for LTX motion clips)
result_prest, prest_log = run_confessional("prest")
print("Prest clip manifest:", prest_log)

KeyError: 'prest'

## Cell 7 - Smoke test: Prest confessional (run first, 3 beats ~3-5 min on Kaggle T4)

## Cell 8 - Run all six confessionalals (after smoke test passes)
## One cell -> comment out all but one if iterating.

In [ ]:
if __name__ == "__main__":
    results = {}
    for char_key in CHARACTERS:
        try:
            c, log = run_confessional(char_key)
            results[char_key] = log
            print(f"DONE: {char_key} -> {log}")
        except Exception as e:
            results[char_key] = f"ERROR: {e}"
    print("=== ALL CONFESSIONALS ===", json.dumps(results, indent=2))

# To run on Kaggle: Runtime -> Change runtime type -> GPU T4 x2 -> Run all.
# Estimated cost: ~3 min/character (audio instant on CPU; LivePortrait ~30s; LTX ~2min on T4).